## Load the Configuration

In [1]:
import json
import time
from copy import deepcopy
from pathlib import Path

from netrun.core import Net, NetConfig

# Load the base configuration
config_path = Path("main.netrun.json")
config_data = json.loads(config_path.read_text())

print("Pool configuration from file:")
print(json.dumps(config_data["pools"], indent=2))

Pool configuration from file:
{
  "main": {
    "spec": {
      "type": "main"
    }
  },
  "threads": {
    "spec": {
      "type": "thread",
      "num_workers": 4
    }
  },
  "processes": {
    "spec": {
      "type": "multiprocess",
      "num_processes": 2,
      "threads_per_process": 2
    }
  },
  "remote": {
    "spec": {
      "type": "remote",
      "url": "ws://127.0.0.1:18765",
      "worker_name": "execution_manager",
      "num_processes": 1,
      "threads_per_process": 1
    }
  }
}


## Start the Remote Pool Server

The `remote` pool in the config needs a server running at `ws://127.0.0.1:18765`.
We use `Net.serve_pool()` which creates a server with the correct func_preprocessor
from the config, enabling factory-based nodes on remote workers.

In [2]:
pool_ctx = Net.serve_pool(config_path, "127.0.0.1", 18765)
await pool_ctx.__aenter__()
print("Remote pool server running on ws://127.0.0.1:18765")

Remote pool server running on ws://127.0.0.1:18765


## Run net

In [3]:
async with Net.from_file(config_path) as net:
    net.nodes['in_main_pool'].inject({
        "start": 0,
        "stop": 100,
    })
    net.nodes['in_thread_pool'].inject({
        "start": 100,
        "stop": 200,
    })
    net.nodes['in_process_pool'].inject({
        "start": 200,
        "stop": 300,
    })
    net.nodes['in_remote_pool'].inject({
        "start": 300,
        "stop": 400,
    })
    net.nodes['in_all_pools'].inject({
        "start": 400,
        "stop": 500,
    });

    made_progress, net_events = await net.run_until_blocked()
    assert made_progress
    res = net.flush_all_output_queues()

  + Exception Group Traceback (most recent call last):
  |   File "/Users/lukas/dev/20260113_w3pmcj__netrun2/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/vp/qt6bq0j54k7cqzzm500hcn3h0000gn/T/ipykernel_52887/918999870.py", line 23, in <module>
  |     made_progress, net_events = await net.run_until_blocked()
  |                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/lukas/dev/20260113_w3pmcj__netrun2/netrun/src/netrun/net/_net/_net.py", line 1220, in run_until_blocked
  |     made_progress, events = await self.run_step(auto_start_epochs=auto_start_epochs)
  |                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/lukas/dev/20260113_w3pmcj__netrun2/netrun/src/netrun/net/_net/_net.py", line 1200, in run_step
  |     raise ExceptionGroup("Multiple epoch failures", exceptions)
  | Ex

# Results

In [ ]:
primes = [v for vals in res.values() for v in vals[0]]
primes.sort()
print("Primes found:")
print(", ".join(map(str, primes)))

Primes found:
2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311, 313, 317, 331, 337, 347, 349, 353, 359, 367, 373, 379, 383, 389, 397, 401, 409, 419, 421, 431, 433, 439, 443, 449, 457, 461, 463, 467, 479, 487, 491, 499


# Logs

In [ ]:
net.print_all_logs()

=== in_main_pool ===
--- Epoch 01KGSYF8B4ZR4QRV3DN92EZJFS ---
[17:02:24.101] Finding all primes between 0 and 100
[17:02:24.101] Found 25 primes: 2...97
=== in_process_pool ===
--- Epoch 01KGSYF8B4YMHQS50X9KWRZWG8 ---
[17:02:24.102] Finding all primes between 200 and 300
[17:02:24.102] Found 16 primes: 211...293
=== in_remote_pool ===
--- Epoch 01KGSYF8B499T2WYA7TB08PW4H ---
[17:02:24.107] Finding all primes between 300 and 400
[17:02:24.108] Found 16 primes: 307...397
=== in_thread_pool ===
--- Epoch 01KGSYF8B4N7WRZZJHYKAM1QFH ---
[17:02:24.101] Finding all primes between 100 and 200
[17:02:24.101] Found 21 primes: 101...199
=== in_all_pools ===
--- Epoch 01KGSYF8B4WYFM5CH534VR2587 ---
[17:02:24.103] Finding all primes between 400 and 500
[17:02:24.103] Found 17 primes: 401...499


## Stop the Remote Pool Server

In [ ]:
await pool_ctx.__aexit__(None, None, None)
print("Remote pool server stopped.")